# Classify Tree Species at CLBJ with Random Forest and Field Data in the Earth Engine Python API

This tutorial classifies woody plant species and three land cover types (grassland, water, and shade) at the NEON Lyndon B. Johnson National Grassland (CLBJ, domain D11) site using 2022 NEON Airborne Observation Platform (AOP) hyperspectral reflectance and lidar canopy height data. It demonstrates how to build training data from NEON vegetation structure field observations and train a random forest classifier in Google Earth Engine (GEE).

This is Phase 2 of the tutorial series. It improves on Phase 1 by:
- Using 2022 AOP data from the BRDF-corrected reflectance collection (`HSI_REFL/002`)
- Building training data directly from NEON vegetation structure (VST) field data (DP1.10098.001)
- Demonstrating the full workflow from field data download to classification

Phase 3 will address remaining methodological limitations including spatial autocorrelation and class imbalance.

## Background Tutorials

This tutorial builds on methods and tools from the following NEON tutorials:

* [Using neonDataStackR to access NEON data](https://www.neonscience.org/resources/learning-hub/tutorials/neondatastackr)
* [Work with NEON AOP reflectance data in Python with geemap](https://www.neonscience.org/resources/learning-hub/tutorials/aop-refl-py-geemap)
* [Classify tree species with NEON AOP data and random forest classification in Google Earth Engine](https://www.neonscience.org/resources/learning-hub/tutorials/aop-gee-random-forest-classification)

## Learning Objectives

After completing this tutorial, you will be able to:

* Download and process NEON vegetation structure data using the NEON API
* Compute stem positions from plot-based measurements using the geoNEON formula
* Generate non-tree training classes using threshold-based methods
* Load and preprocess NEON AOP reflectance and CHM data in Google Earth Engine
* Train and evaluate a random forest classifier for tree species mapping
* Interpret out-of-bag error estimates and test set accuracy metrics

## Data Products and Study Site

The notebook uses the following NEON data products. Earth Engine hosts the AOP products as image collections and the site boundaries as feature collections.

| NEON data product | ID | Earth Engine asset |
|---|---|---|
| Spectrometer orthorectified surface directional reflectance - mosaic (BRDF-corrected) | [DP3.30006.002](https://data.neonscience.org/data-products/DP3.30006.002) | `projects/neon-prod-earthengine/assets/HSI_REFL/002/2022_CLBJ_6` |
| Ecosystem structure (canopy height model, CHM) | [DP3.30015.001](https://data.neonscience.org/data-products/DP3.30015.001) | `projects/neon-prod-earthengine/assets/CHM/001` |
| Woody plant vegetation structure | [DP1.10098.001](https://data.neonscience.org/data-products/DP1.10098.001) | Downloaded via NEON API |
| Tower airshed | | `projects/neon-prod-earthengine/assets/Feature_Collections/90percent_footprint` |
| Terrestrial Observation System boundary | | `projects/neon-prod-earthengine/assets/Feature_Collections/terrestrialSamplingBoundaries` |

**Site:** CLBJ, Lyndon B. Johnson National Grassland, NEON domain D11 (Texas).

**Year:** 2022. The AOP flight was in late spring. We use VST data from April and May 2022 to match the phenology.

**Reference data:** The training points are stem locations from VST field measurements, converted to geographic coordinates using the geoNEON formula. We filter to live trees with full sun exposure and height ≥ 3 m. Non-tree classes (grassland, water, shade) are generated using threshold-based rules on NDVI, brightness, and CHM.

## Setup

Import the packages and initialize Earth Engine. Earth Engine requires a Google Cloud project for every session.

In [1]:
import functools
import os
import math

import ee
import geemap
import neonutilities as nu
import pandas as pd
import numpy as np
import pyproj
import requests

# Change this to the ID of your own Google Cloud project that is registered for Earth Engine.
PROJECT_ID = 'neon-sandbox-dataflow-ee'

ee.Initialize(project=PROJECT_ID)

## Workflow

1. Load boundaries and imagery from Earth Engine
2. Download and process VST field data to create tree species training points
3. Generate non-tree classes using threshold-based methods
4. Split data into training and test sets and train a random forest classifier
5. Evaluate model accuracy on the independent test set

## Step 1: Load Boundaries and Imagery

First, load the site boundaries and 2022 AOP data from Earth Engine. The airshed defines the study area, and the TOS boundary shows the terrestrial sampling extent.

In [2]:
# Specify the CLBJ location (longitude, latitude)
geo = ee.Geometry.Point([-97.5706464, 33.4045729])

# Tower airshed boundary (90% footprint) - note capital SiteID
airshed_collection = ee.FeatureCollection('projects/neon-prod-earthengine/assets/Feature_Collections/90percent_footprint')
clbj_airshed = airshed_collection.filter(ee.Filter.eq('SiteID', 'CLBJ'))

# Union the two polygons into one boundary
CLBJ_Airshed = clbj_airshed.union()

# Terrestrial Observation System (TOS) boundary - note lowercase siteID
tos_collection = ee.FeatureCollection('projects/neon-prod-earthengine/assets/Feature_Collections/terrestrialSamplingBoundaries')
CLBJ_TOS = tos_collection.filter(ee.Filter.eq('siteID', 'CLBJ'))

### Canopy Height Model (CHM)

Load the 2022 CHM and mask negative values (no-data).

In [3]:
# Load CHM for 2022 at CLBJ
chm_collection = ee.ImageCollection('projects/neon-prod-earthengine/assets/CHM/001')
CLBJ_CHM2022 = (chm_collection
                .filterDate('2022-01-01', '2022-12-31')
                .filterBounds(geo)
                .first())

# The band is named 'CHM'. Mask values < 0 (no-data).
CLBJ_CHM2022mask = CLBJ_CHM2022.select('CHM').updateMask(CLBJ_CHM2022.select('CHM').gte(0))

print('CHM band:', CLBJ_CHM2022mask.bandNames().getInfo())

CHM band: ['CHM']


### Surface Directional Reflectance (BRDF-corrected)

Load the 2022 BRDF-corrected reflectance mosaic. The image has 426 bands (B001-B426) stored as integers scaled by a factor of 10,000. We divide by the scaling factor to get fractional reflectance (0-1).

Water vapor absorbs most light in certain wavelength ranges, making those bands noisy. We remove:
- Bands 195-205 (water absorption)
- Bands 287-310 (water absorption)
- Bands 416-425 (noisy edge bands)

This leaves 381 bands for the classification.

In [4]:
# Load 2022 reflectance
CLBJ_SDR2022 = ee.Image('projects/neon-prod-earthengine/assets/HSI_REFL/002/2022_CLBJ_6')

# Select bands B001-B426 and apply the scaling factor
band_names_426 = [f'B{i:03d}' for i in range(1, 427)]
CLBJ_SDR2022_bands = CLBJ_SDR2022.select(band_names_426)

# Get the SCALING_FACTOR property and divide
scaling_factor = ee.Number(CLBJ_SDR2022.get('SCALING_FACTOR'))
CLBJ_SDR2022_scaled = CLBJ_SDR2022_bands.divide(scaling_factor)

print('Scaling factor:', scaling_factor.getInfo())

Scaling factor: 10000


In [5]:
# Remove water absorption and noisy bands
bandsToRemove = [f'B{i:03d}' for i in [*range(195, 206), *range(287, 311), *range(416, 426)]]
all_band_names = ee.List([f'B{i:03d}' for i in range(1, 427)])
bandsToKeep = all_band_names.removeAll(bandsToRemove)

CLBJ_SDR2022subset = CLBJ_SDR2022_scaled.select(bandsToKeep)

print('Bands removed:', len(bandsToRemove))
print('Bands kept:', bandsToKeep.size().getInfo())  # 381 expected

Bands removed: 45
Bands kept: 381


In [6]:
# Display version: select RGB bands and mask no-data
# For 2022 data, use bands B053 (red), B035 (green), B019 (blue)
CLBJ_SDR2022mask = CLBJ_SDR2022_scaled.select(['B053', 'B035', 'B019'])

# Mask negative values (no-data indicators)
CLBJ_SDR2022mask = CLBJ_SDR2022mask.updateMask(CLBJ_SDR2022mask.select('B053').gte(0))

### Create the Predictor Composite

Combine the 381 reflectance bands and the CHM into a single composite image. This composite provides all the predictor variables for the random forest classifier.

In [7]:
# Combine the 381 reflectance bands and 1 CHM band (382 bands total)
CLBJ_composite = CLBJ_SDR2022subset.addBands(CLBJ_CHM2022mask)

# Clip to the airshed boundary
CLBJ_composite_airshed = CLBJ_composite.clip(CLBJ_Airshed)

print('Composite bands:', CLBJ_composite_airshed.bandNames().size().getInfo())  # 382 expected

Composite bands: 382


### Map 1: Input Data

Display the reflectance RGB composite, CHM, and boundaries on an interactive map.

In [8]:
map_inputs = geemap.Map()

# RGB composite (adjust min/max for 2022 data - fractional reflectance)
map_inputs.addLayer(CLBJ_SDR2022mask, {'min': 0.005, 'max': 0.10}, 'CLBJ SDR 2022 RGB', False)
map_inputs.addLayer(CLBJ_CHM2022mask, {'min': 0, 'max': 33, 'palette': ['green', 'yellow', 'red']}, 'CLBJ CHM 2022', False)
map_inputs.addLayer(CLBJ_composite_airshed,
                    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
                    'CLBJ-Airshed Composite 2022', True)

# Boundaries with transparent fill
map_inputs.addLayer(CLBJ_TOS.style(width=3, color='blue', fillColor='#00000000'), {}, 'CLBJ TOS', False)
map_inputs.addLayer(CLBJ_Airshed.style(width=3, color='white', fillColor='#00000000'), {}, 'CLBJ Airshed', False)

map_inputs.centerObject(geo, 16)
map_inputs

Map(center=[33.404572900000005, -97.5706464], controls=(WidgetControl(options=['position', 'transparent_bg'], …

## Step 2: Download and Process VST Field Data

We'll download NEON vegetation structure (VST) data from April and May 2022 to build our tree species training points. The VST data includes stem measurements, species identifications, and plot-relative positions that we'll convert to geographic coordinates.

### Download VST Data

The NEON API requires a token for data downloads. Set your token as an environment variable `NEON_TOKEN` before running this notebook. You can obtain a token from your NEON user account.

In [9]:
# Read NEON API token from environment variable
neon_token = os.environ.get('NEON_TOKEN')
if not neon_token:
    raise ValueError('NEON_TOKEN environment variable is not set. '
                     'Please set it to your NEON API token before running this notebook.')

print('NEON token found (not displayed for security)')

NEON token found (not displayed for security)


In [10]:
# Download VST data for CLBJ, April-May 2022
# This may take a minute or two
vst_data = nu.load_by_product(
    dpid='DP1.10098.001',
    site='CLBJ',
    startdate='2022-04',
    enddate='2022-05',
    package='basic',
    check_size=False,
    token=neon_token
)

print('VST data downloaded')

Finding available files
100%|██████████| 2/2 [00:01<00:00,  1.53it/s]
Stacking data files
100%|██████████| 5/5 [00:00<00:00, 13.62it/s]


VST data downloaded


### Extract VST Tables

The VST data product includes several tables. We need:
- `vst_mappingandtagging`: plot-relative stem positions and identifiers
- `vst_apparentindividual`: plant status, canopy position, and height

In [11]:
# Extract the relevant tables
mapping_tagging = vst_data['vst_mappingandtagging']
apparent_individual = vst_data['vst_apparentindividual']

print(f'Mapping and tagging records: {len(mapping_tagging)}')
print(f'Apparent individual records: {len(apparent_individual)}')

Mapping and tagging records: 3589
Apparent individual records: 156


In [12]:
# Display key columns from apparent individual table
individual_cols = ['individualID', 'plantStatus', 'canopyPosition', 'height']
print('Apparent Individual Table (sample rows):')
print(f'Shows plant characteristics: status, canopy position, and height\n')
display(apparent_individual[individual_cols].head(10))

print(f'\nCanopy positions present:')
print(apparent_individual['canopyPosition'].value_counts())
print(f'\nHeight range: {apparent_individual["height"].min():.1f} - {apparent_individual["height"].max():.1f} m')

Apparent Individual Table (sample rows):
Shows plant characteristics: status, canopy position, and height



,individualID,plantStatus,canopyPosition,height
0,NEON.PLA.D11.CLBJ.02152,"Live, broken bole",Partially shaded,3.4
1,NEON.PLA.D11.CLBJ.03192,"Live, physically damaged",Partially shaded,0.8
2,NEON.PLA.D11.CLBJ.03274,"Live, physically damaged",,NaN
3,NEON.PLA.D11.CLBJ.03265,"Live, physically damaged",,NaN
4,NEON.PLA.D11.CLBJ.03284,"Live, physically damaged",,NaN
5,NEON.PLA.D11.CLBJ.01011,"Live, physically damaged",,NaN
6,NEON.PLA.D11.CLBJ.01012,"Live, physically damaged",,NaN
7,NEON.PLA.D11.CLBJ.02176,Live,Partially shaded,8.0
8,NEON.PLA.D11.CLBJ.03290,"Live, physically damaged",Full sun,1.0
9,NEON.PLA.D11.CLBJ.03378,Live,Mostly shaded,1.1



Canopy positions present:
canopyPosition
Mostly shaded       54
                    41
Partially shaded    39
Full shade          18
Full sun             4
Name: count, dtype: int64

Height range: 0.5 - 10.3 m


In [13]:
# Display key columns from mapping and tagging table
mapping_cols = ['plotID', 'pointID', 'individualID', 'taxonID', 'stemDistance', 'stemAzimuth']
print('Mapping and Tagging Table (sample rows):')
print(f'Shows plot locations and stem positions relative to plot corner points\n')
display(mapping_tagging[mapping_cols].head(10))

print(f'\nMissing coordinates: {mapping_tagging["stemDistance"].isna().sum()} of {len(mapping_tagging)} records ({100*mapping_tagging["stemDistance"].isna().sum()/len(mapping_tagging):.1f}%)')
print(f'Species present: {mapping_tagging["taxonID"].nunique()} unique taxa')

Mapping and Tagging Table (sample rows):
Shows plot locations and stem positions relative to plot corner points



,plotID,pointID,individualID,taxonID,stemDistance,stemAzimuth
0,CLBJ_029,,NEON.PLA.D11.CLBJ.00755,ULCR,NaN,NaN
1,CLBJ_029,49,NEON.PLA.D11.CLBJ.00752,QUERC,6.7,167.2
2,CLBJ_029,49,NEON.PLA.D11.CLBJ.00762,ULCR,8.5,155.7
3,CLBJ_029,,NEON.PLA.D11.CLBJ.00754,ULCR,NaN,NaN
4,CLBJ_029,,NEON.PLA.D11.CLBJ.00769,ULCR,NaN,NaN
5,CLBJ_029,,NEON.PLA.D11.CLBJ.00753,ULCR,NaN,NaN
6,CLBJ_029,,NEON.PLA.D11.CLBJ.00782,CELA,NaN,NaN
7,CLBJ_029,,NEON.PLA.D11.CLBJ.00760,ULMUS,NaN,NaN
8,CLBJ_029,,NEON.PLA.D11.CLBJ.00763,ULCR,NaN,NaN
9,CLBJ_029,,NEON.PLA.D11.CLBJ.00768,QUERC,NaN,NaN



Missing coordinates: 2757 of 3589 records (76.8%)
Species present: 40 unique taxa


### Examine VST Table Structure

Before processing the VST data, let's examine the table structure to understand what columns are available and what the data looks like. This helps us understand what we're working with when we filter and process the data.

### Compute Stem Geographic Coordinates

VST data record stem positions relative to plot corner points using `stemDistance` and `stemAzimuth`. We use the geoNEON formula to convert these to absolute coordinates:

1. Query the NEON location API for each plot point's UTM coordinates
2. Apply the geoNEON formula: 
   - `adjEasting = pointEasting + stemDistance × sin(stemAzimuth × π/180)`
   - `adjNorthing = pointNorthing + stemDistance × cos(stemAzimuth × π/180)`
3. Convert from UTM Zone 14N (EPSG:32614) to WGS84 lon/lat (EPSG:4326)

In [14]:
# Get unique plot/point combinations
plot_points = mapping_tagging[['plotID', 'pointID']].drop_duplicates()
print(f'Unique plot-point combinations: {len(plot_points)}')

Unique plot-point combinations: 214


In [15]:
# Function to query NEON location API for point coordinates
def get_point_location(plot_id, point_id, cache={}):
    """Get UTM coordinates for a plot point from the NEON location API.
    
    Args:
        plot_id: Plot identifier (e.g., 'CLBJ_001')
        point_id: Point identifier (e.g., '41')
        cache: Dictionary to cache results
    
    Returns:
        Tuple of (easting, northing) or (None, None) if not found
    """
    key = (plot_id, point_id)
    if key in cache:
        return cache[key]
    
    # Construct the named location
    named_location = f"{plot_id}.basePlot.vst.{point_id}"
    url = f"https://data.neonscience.org/api/v0/locations/{named_location}"
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            easting = data['data']['locationUtmEasting']
            northing = data['data']['locationUtmNorthing']
            cache[key] = (easting, northing)
            return (easting, northing)
        else:
            cache[key] = (None, None)
            return (None, None)
    except Exception as e:
        print(f"Error fetching {named_location}: {e}")
        cache[key] = (None, None)
        return (None, None)

print('Location API query function defined')

Location API query function defined


In [16]:
# Query point locations (this may take a few minutes due to API rate limits)
print('Querying NEON location API for plot point coordinates...')
location_cache = {}

for idx, row in plot_points.iterrows():
    get_point_location(row['plotID'], row['pointID'], location_cache)
    if (idx + 1) % 10 == 0:
        print(f'  Queried {idx + 1}/{len(plot_points)} locations')

print(f'Location query complete: {len(location_cache)} unique locations cached')

Querying NEON location API for plot point coordinates...
  Queried 20/214 locations
  Queried 70/214 locations
  Queried 160/214 locations
  Queried 210/214 locations
  Queried 230/214 locations
  Queried 280/214 locations
  Queried 370/214 locations
  Queried 400/214 locations
  Queried 460/214 locations
  Queried 480/214 locations
  Queried 530/214 locations
  Queried 590/214 locations
  Queried 660/214 locations
  Queried 680/214 locations
  Queried 720/214 locations
  Queried 830/214 locations
  Queried 940/214 locations
  Queried 950/214 locations
  Queried 1090/214 locations
  Queried 1180/214 locations
  Queried 1190/214 locations
  Queried 1300/214 locations
  Queried 1340/214 locations
  Queried 1430/214 locations
  Queried 1560/214 locations
  Queried 1600/214 locations
  Queried 1810/214 locations
  Queried 2060/214 locations
Location query complete: 214 unique locations cached


In [18]:
# Apply geoNEON formula to compute stem positions
def compute_stem_position(row):
    """Compute stem UTM position using the geoNEON formula."""
    point_easting, point_northing = location_cache.get(
        (row['plotID'], row['pointID']),
        (None, None)
    )
    
    if point_easting is None or pd.isna(row['stemDistance']) or pd.isna(row['stemAzimuth']):
        return pd.Series({'stemEasting': None, 'stemNorthing': None})
    
    # geoNEON formula
    azimuth_rad = row['stemAzimuth'] * math.pi / 180
    stem_easting = point_easting + row['stemDistance'] * math.sin(azimuth_rad)
    stem_northing = point_northing + row['stemDistance'] * math.cos(azimuth_rad)
    
    return pd.Series({'stemEasting': stem_easting, 'stemNorthing': stem_northing})

# Apply to mapping_tagging table
mapping_tagging[['stemEasting', 'stemNorthing']] = mapping_tagging.apply(
    compute_stem_position,
    axis=1
)

print(f'Stem positions computed for {mapping_tagging["stemEasting"].notna().sum()} stems')

Stem positions computed for 830 stems


In [19]:
# Convert UTM Zone 14N to WGS84 lon/lat
transformer = pyproj.Transformer.from_crs('EPSG:32614', 'EPSG:4326', always_xy=True)

def utm_to_lonlat(row):
    """Convert UTM coordinates to lon/lat."""
    if pd.isna(row['stemEasting']) or pd.isna(row['stemNorthing']):
        return pd.Series({'longitude': None, 'latitude': None})
    
    lon, lat = transformer.transform(row['stemEasting'], row['stemNorthing'])
    return pd.Series({'longitude': lon, 'latitude': lat})

mapping_tagging[['longitude', 'latitude']] = mapping_tagging.apply(utm_to_lonlat, axis=1)

print(f'Geographic coordinates computed for {mapping_tagging["longitude"].notna().sum()} stems')

Geographic coordinates computed for 830 stems


### Join Tables and Filter to Visible Trees

Join the mapping/tagging table with the apparent individual table to get plant status and canopy characteristics. Filter to:
- Live trees (`plantStatus` contains "Live")
- Full sun exposure (`canopyPosition` in ["Full sun", "Open grown"])
- Height ≥ 3 m (visible from air)
- Valid geographic coordinates

In [20]:
# Join tables on individualID
vst_joined = mapping_tagging.merge(
    apparent_individual[['individualID', 'plantStatus', 'canopyPosition', 'height']],
    on='individualID',
    how='left'
)

print(f'Joined table: {len(vst_joined)} records')

Joined table: 3616 records


In [21]:
# Relaxed filters for 2022 CLBJ data
vst_filtered = vst_joined[
    (vst_joined['stemDistance'].notna()) &
    (vst_joined['stemAzimuth'].notna()) &
    (vst_joined['plantStatus'].str.contains('Live', na=False)) &
    (vst_joined['canopyPosition'].isin(['Full sun', 'Partially shaded', 'Mostly shaded'])) &
    (vst_joined['height'] >= 1.0)
].copy()

print(f"Filtered trees: {len(vst_filtered)} stems")
print(f"\nSpecies counts:")
print(vst_filtered['taxonID'].value_counts())

Filtered trees: 17 stems

Species counts:
taxonID
QUST     9
JUVI     4
ULCR     3
QUMA3    1
Name: count, dtype: int64


In [22]:
# Drop QUMA3 (only 1 sample, can't split for train/test)
vst_filtered = vst_filtered[vst_filtered['taxonID'] != 'QUMA3'].copy()

print(f"Final tree samples: {len(vst_filtered)} stems")
print(vst_filtered['taxonID'].value_counts())

Final tree samples: 16 stems
taxonID
QUST    9
JUVI    4
ULCR    3
Name: count, dtype: int64


## Step 3: Generate Non-Tree Classes

VST data only includes trees. We need to add training points for three additional land cover classes:
- **GRSS (Grassland)**: Low vegetation with high greenness
- **WATR (Water)**: Very low NDVI and low brightness
- **SHADE (Shade)**: Dark areas under tall canopy

We'll use threshold-based rules on NDVI, brightness, and CHM to identify these areas.

### Compute NDVI and Brightness

In [23]:
# Compute NDVI from scaled reflectance
# NDVI = (NIR - Red) / (NIR + Red)
# B090 is NIR (~800nm), B054 is Red (~670nm)
nir = CLBJ_SDR2022_scaled.select('B090')
red = CLBJ_SDR2022_scaled.select('B054')
ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

# Compute brightness as mean of visible bands (B019-B060)
visible_bands = [f'B{i:03d}' for i in range(19, 61)]
brightness = CLBJ_SDR2022_scaled.select(visible_bands).reduce(ee.Reducer.mean()).rename('brightness')

print('NDVI and brightness computed')

NDVI and brightness computed


### Define Threshold Rules

Create masks for each non-tree class based on spectral and structural characteristics.

In [24]:
# Get CHM for threshold calculations
chm = CLBJ_CHM2022mask.select('CHM')

# GRSS (grassland): high NDVI, low-medium height
grss_mask = chm.lt(4.0).And(ndvi.gt(0.4))

# WATR (water): low NDVI, low brightness
watr_mask = ndvi.lt(0.3).And(brightness.lt(0.03))

# SHADE (shadow): very low brightness
shade_mask = brightness.lt(0.025)

print('Threshold masks created')

Threshold masks created


In [25]:
# Sample points from each mask (10-15 points per class)
# Use stratifiedSample with a dummy class property

# Create single-band images for sampling
grss_image = grss_mask.selfMask().clip(CLBJ_Airshed)
watr_image = watr_mask.selfMask().clip(CLBJ_Airshed)
shade_image = shade_mask.selfMask().clip(CLBJ_Airshed)

# Sample points
grss_points = grss_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=42,
    geometries=True
)

watr_points = watr_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=43,
    geometries=True
)

shade_points = shade_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=44,
    geometries=True
)

print('Non-tree points sampled')
print(f'  GRSS: {grss_points.size().getInfo()} points')
print(f'  WATR: {watr_points.size().getInfo()} points')
print(f'  SHADE: {shade_points.size().getInfo()} points')

Non-tree points sampled
  GRSS: 2 points
  WATR: 0 points
  SHADE: 2 points


In [26]:
# Manually selected non-tree training points (from visual inspection of 2022 image)

# Water points (from ponds/lakes in the airshed)
watr_coords = [
    [-97.57458, 33.38499],
    [-97.57383, 33.38580],
    [-97.57270, 33.38593],
    [-97.57339, 33.38527],
    [-97.57251, 33.38495],
    [-97.57374, 33.38454],
    [-97.57040, 33.40716],
    [-97.56965, 33.40669],
    [-97.56856, 33.40553],
    [-97.57010, 33.40651]
]

# Grassland points (open areas, low vegetation)
grss_coords = [
    [-97.57794, 33.40790],
    [-97.57343, 33.41121],
    [-97.54877, 33.41231],
    [-97.55444, 33.41417],
    [-97.61723, 33.41045],
    [-97.60071, 33.41111],
    [-97.65015, 33.41466],
    [-97.62499, 33.41824],
    [-97.61812, 33.41852],
    [-97.61755, 33.41786]
]

# Shade points (shadows under dense canopy)
shade_coords = [
    [-97.62058, 33.41911],
    [-97.62008, 33.41934],
    [-97.62239, 33.41942],
    [-97.62245, 33.41968],
    [-97.61363, 33.42193],
    [-97.60779, 33.41329],
    [-97.60730, 33.41298],
    [-97.60689, 33.41319],
    [-97.60876, 33.41297],
    [-97.60916, 33.41316]
]

# Create FeatureCollections with class labels
grss_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 7, 'className': 'GRSS'}) 
    for c in grss_coords
])

watr_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 8, 'className': 'WATR'}) 
    for c in watr_coords
])

shade_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 9, 'className': 'SHADE'}) 
    for c in shade_coords
])

print('Non-tree training points created:')
print(f'  GRSS: {len(grss_coords)} points')
print(f'  WATR: {len(watr_coords)} points')
print(f'  SHADE: {len(shade_coords)} points')
print(f'  Total non-tree: {len(grss_coords) + len(watr_coords) + len(shade_coords)} points')

Non-tree training points created:
  GRSS: 10 points
  WATR: 10 points
  SHADE: 10 points
  Total non-tree: 30 points


In [27]:
# Create map to visualize all reference points
points_map = geemap.Map()

# Add RGB basemap
points_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

# Add non-tree points with distinct colors
points_map.addLayer(
    grss_points.style(pointSize=5, color='00FF00'),  # Bright green
    {},
    'GRSS (grassland)',
    True
)

points_map.addLayer(
    watr_points.style(pointSize=5, color='0000FF'),  # Blue
    {},
    'WATR (water)',
    True
)

points_map.addLayer(
    shade_points.style(pointSize=5, color='000000'),  # Black
    {},
    'SHADE (shadow)',
    True
)

# Add airshed boundary
points_map.addLayer(
    CLBJ_Airshed.style(fillColor='00000000', color='FFFFFF', width=2),
    {},
    'Airshed',
    True
)

points_map.centerObject(CLBJ_Airshed, 13)
points_map

Map(center=[33.40123000138371, -97.57000034548355], controls=(WidgetControl(options=['position', 'transparent_…

### Visualize Threshold Masks

Display the threshold masks on a map to verify they capture the intended land cover types.

In [28]:
map_thresholds = geemap.Map()

# Base layer
map_thresholds.addLayer(CLBJ_SDR2022mask, {'min': 0.005, 'max': 0.10}, 'CLBJ SDR 2022 RGB', True)

# Threshold masks
map_thresholds.addLayer(grss_mask.selfMask(), {'palette': ['lightgreen']}, 'GRSS mask', False)
map_thresholds.addLayer(watr_mask.selfMask(), {'palette': ['blue']}, 'WATR mask', False)
map_thresholds.addLayer(shade_mask.selfMask(), {'palette': ['black']}, 'SHADE mask', False)

map_thresholds.centerObject(geo, 16)
map_thresholds

Map(center=[33.404572900000005, -97.5706464], controls=(WidgetControl(options=['position', 'transparent_bg'], …

### Build Complete Training FeatureCollection

Combine the tree species points from VST with the non-tree points. Assign numeric class labels (`taxonIDnum`) dynamically based on the species present in the data.

In [29]:
# Create class lookup dictionary from VST species
unique_species = vst_filtered['taxonID'].unique()
class_lookup = {species: idx for idx, species in enumerate(sorted(unique_species))}

# Add non-tree classes
max_tree_class = len(class_lookup)
class_lookup['GRSS'] = max_tree_class
class_lookup['WATR'] = max_tree_class + 1
class_lookup['SHADE'] = max_tree_class + 2

print('Class lookup table:')
for taxon, num in sorted(class_lookup.items(), key=lambda x: x[1]):
    print(f'  {taxon}: {num}')

Class lookup table:
  JUVI: 0
  QUST: 1
  ULCR: 2
  GRSS: 3
  WATR: 4
  SHADE: 5


In [30]:
# Add taxonIDnum to tree data
vst_filtered['taxonIDnum'] = vst_filtered['taxonID'].map(class_lookup)

# Build tree FeatureCollection with class assignments
class_lookup = {'JUVI': 0, 'QUST': 1, 'ULCR': 2}

tree_features = []
for idx, row in vst_filtered.iterrows():
    geom = ee.Geometry.Point([row['longitude'], row['latitude']])
    
    props = {
        'taxonID': row['taxonID'],
        'class': class_lookup[row['taxonID']],  # Add this line!
        'height': float(row['height']),
        'individualID': row['individualID']
    }
    
    tree_features.append(ee.Feature(geom, props))

tree_fc = ee.FeatureCollection(tree_features)
print(f"Tree FeatureCollection: {len(tree_features)} features")

# Verify class assignments
for taxon, class_num in class_lookup.items():
    count = tree_fc.filter(ee.Filter.eq('class', class_num)).size().getInfo()
    print(f"  {taxon} (class {class_num}): {count} features")

Tree FeatureCollection: 16 features
  JUVI (class 0): 4 features
  QUST (class 1): 9 features
  ULCR (class 2): 3 features


In [31]:
# Non-tree points already have 'class' property (7, 8, 9)
# Just add taxonID for consistency with tree points

grss_fc = grss_points.map(lambda f: f.set('taxonID', 'GRSS'))
watr_fc = watr_points.map(lambda f: f.set('taxonID', 'WATR'))
shade_fc = shade_points.map(lambda f: f.set('taxonID', 'SHADE'))

# Merge all classes
all_points = tree_fc.merge(grss_fc).merge(watr_fc).merge(shade_fc)

print(f'Total training points: {all_points.size().getInfo()}')

# Verify class distribution
print('\nPoints per class:')
for class_num in [0, 1, 2, 7, 8, 9]:
    count = all_points.filter(ee.Filter.eq('class', class_num)).size().getInfo()
    class_names = {0: 'JUVI', 1: 'QUST', 2: 'ULCR', 7: 'GRSS', 8: 'WATR', 9: 'SHADE'}
    print(f'  {class_names[class_num]} (class {class_num}): {count} points')

Total training points: 46

Points per class:
  JUVI (class 0): 4 points
  QUST (class 1): 9 points
  ULCR (class 2): 3 points
  GRSS (class 7): 10 points
  WATR (class 8): 10 points
  SHADE (class 9): 10 points


In [32]:
# Count points by class (using 'class' property, not 'taxonIDnum')
class_lookup_all = {
    'JUVI': 0, 'QUST': 1, 'ULCR': 2,
    'GRSS': 7, 'WATR': 8, 'SHADE': 9
}

class_counts = {}
for taxon, class_num in class_lookup_all.items():
    count = all_points.filter(ee.Filter.eq('class', class_num)).size().getInfo()
    class_counts[taxon] = count

counts_df = pd.DataFrame.from_dict(class_counts, orient='index', columns=['Count'])
counts_df.index.name = 'Class'
counts_df = counts_df.sort_values('Count', ascending=False)
print('\nPoints per class:')
display(counts_df)


Points per class:


,Count
Class,
WATR,10
GRSS,10
SHADE,10
QUST,9
JUVI,4
ULCR,3


### Map 2: Reference Points by Class

Display all training points colored by class.

In [33]:
# Map 2: Reference Points (corrected class numbers)
ref_points_map = geemap.Map()

ref_points_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

# Classes with correct numbers (0,1,2,7,8,9)
classes = [
    (0, 'JUVI', 'yellow'),
    (1, 'QUST', 'darkgreen'),
    (2, 'ULCR', 'purple'),
    (7, 'GRSS', 'lightgreen'),
    (8, 'WATR', 'blue'),
    (9, 'SHADE', 'black')
]

for class_num, name, color in classes:
    class_points = all_points.filter(ee.Filter.eq('class', class_num))
    styled = class_points.map(lambda f: f.set('style', {
        'color': color,
        'pointShape': 'diamond',
        'pointSize': 5,
        'width': 2,
        'fillColor': '00000000',
    }))
    ref_points_map.addLayer(styled.style(styleProperty='style'), {}, name, True)

ref_points_map.centerObject(all_points, 14)
ref_points_map

Map(center=[33.408639796259635, -97.59489802976658], controls=(WidgetControl(options=['position', 'transparent…

## Step 4: Train and Test Split

Split the reference points into training (80%) and test (20%) sets using stratified random sampling. The seed ensures reproducibility.

In [34]:
# Add random column for stratified split
with_random = all_points.randomColumn(seed=1)

# Split 80/20
training = with_random.filter(ee.Filter.lt('random', 0.80))
test = with_random.filter(ee.Filter.gte('random', 0.80))

print(f'Training points: {training.size().getInfo()}')
print(f'Test points: {test.size().getInfo()}')

Training points: 38
Test points: 8


### Extract Predictor Values at Training Points

Sample the composite image at each training point to extract the spectral signatures and CHM values.

In [35]:
# Sample training points (extract band values)
training_samples = CLBJ_composite.sampleRegions(  # Use CLBJ_composite, not _airshed
    collection=training,
    properties=['class'],
    scale=1,
    tileScale=16
)

sample_count = training_samples.size().getInfo()
print(f'Training samples extracted: {sample_count}')

Training samples extracted: 37


### Train Random Forest Classifier

Train a random forest with 100 trees using all 382 predictor bands.

In [36]:
# Train classifier
classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_samples,
    classProperty='class',
    inputProperties=CLBJ_composite.bandNames()  # Match the composite used
)

print('Classifier trained')

# Classify the full composite
classified = CLBJ_composite.classify(classifier)
print('Image classified')

# Can clip the result for display if needed
classified_airshed = classified.clip(CLBJ_Airshed)

Classifier trained
Image classified


### Out-of-Bag Error

The out-of-bag (OOB) error is a cross-validation estimate from the random forest training. It's more reliable than resubstitution accuracy.

In [37]:
# Get OOB error from classifier explanation
classifier_info = classifier.explain().getInfo()
oob_error = classifier_info.get('outOfBagErrorEstimate', 'Not available')

if oob_error != 'Not available':
    oob_accuracy = 1 - oob_error
    print(f'Out-of-bag error: {oob_error:.4f}')
    print(f'Out-of-bag accuracy: {oob_accuracy:.4f}')
else:
    print('OOB error not available from this classifier')

Out-of-bag error: 0.1622
Out-of-bag accuracy: 0.8378


In [38]:
# Training resubstitution accuracy (for comparison with Phase 1)
train_classified = training_samples.classify(classifier)
train_matrix = train_classified.errorMatrix('taxonIDnum', 'classification')
train_accuracy = train_matrix.accuracy().getInfo()

print(f'Training resubstitution accuracy: {train_accuracy:.4f}')
print('(Note: This is optimistically biased; use OOB or test accuracy instead)')

Training resubstitution accuracy: 0.0000
(Note: This is optimistically biased; use OOB or test accuracy instead)


## Step 5: Evaluate on Test Set

Evaluate the classifier on the independent test set to get an unbiased accuracy estimate.

In [39]:
# Sample test points (use unclipped composite)
test_samples = CLBJ_composite.sampleRegions(  # Use CLBJ_composite, not _airshed
    collection=test,
    properties=['class'],
    scale=1,
    tileScale=16
)

test_count = test_samples.size().getInfo()
print(f'Test samples: {test_count}')

# Classify test samples
test_classified = test_samples.classify(classifier)

# Compute accuracy metrics
error_matrix = test_classified.errorMatrix('class', 'classification')
overall_accuracy = error_matrix.accuracy().getInfo()
kappa = error_matrix.kappa().getInfo()

print(f'\nTest Set Accuracy:')
print(f'Overall Accuracy: {overall_accuracy:.4f}')
print(f'Kappa Coefficient: {kappa:.4f}')

Test samples: 8

Test Set Accuracy:
Overall Accuracy: 0.8750
Kappa Coefficient: 0.8462


### Confusion Matrix and Per-Class Accuracy

In [40]:
# Get confusion matrix
matrix = error_matrix.getInfo()

# Full class mapping for all 6 classes we're using (0,1,2,7,8,9)
class_names_all = {
    0: 'JUVI', 1: 'QUST', 2: 'ULCR',
    7: 'GRSS', 8: 'WATR', 9: 'SHADE'
}

# Matrix is 10x10 (classes 0-9), but we only use 6 classes
# Extract the relevant rows/columns
matrix_array = np.array(matrix)

# Our classes in order
our_classes = [0, 1, 2, 7, 8, 9]
class_labels = [class_names_all[i] for i in our_classes]

# Extract submatrix for our classes only
sub_matrix = matrix_array[np.ix_(our_classes, our_classes)]

# Display confusion matrix
confusion_df = pd.DataFrame(
    sub_matrix.astype(int),
    index=class_labels,
    columns=class_labels
)
confusion_df.index.name = 'Reference'
confusion_df.columns.name = 'Predicted'

print('\nConfusion Matrix:')
display(confusion_df)

print(f'\nTest Set Size: {sub_matrix.sum():.0f} points')
print(f'Correctly Classified: {np.trace(sub_matrix):.0f}')


Confusion Matrix:


Predicted,JUVI,QUST,ULCR,GRSS,WATR,SHADE
Reference,,,,,,
JUVI,1,0,0,0,0,0
QUST,0,1,0,0,0,0
ULCR,1,0,0,0,0,0
GRSS,0,0,0,1,0,0
WATR,0,0,0,0,2,0
SHADE,0,0,0,0,0,2



Test Set Size: 8 points
Correctly Classified: 7


In [ ]:
# Compute per-class accuracy metrics from the confusion matrix
# Producers accuracy (recall) = diagonal / column sum (how many reference samples were correctly classified)
# Users accuracy (precision) = diagonal / row sum (how many predictions were correct)

# Column sums (reference totals)
col_sums = sub_matrix.sum(axis=0)
# Row sums (predicted totals)  
row_sums = sub_matrix.sum(axis=1)
# Diagonal (correct predictions)
diagonal = np.diag(sub_matrix)

# Compute accuracies (avoid division by zero)
producers_acc = np.where(col_sums > 0, diagonal / col_sums, 0)
users_acc = np.where(row_sums > 0, diagonal / row_sums, 0)

# Display per-class accuracy
accuracy_df = pd.DataFrame({
    'Producers accuracy': producers_acc,
    'Users accuracy': users_acc
}, index=class_labels)

print('\nPer-Class Accuracy:')
display(accuracy_df.style.format('{:.2%}'))

### Map 3: Classification Result

Display the classified image with training points overlaid.

In [89]:
# Map 3: Classification Results
results_map = geemap.Map()

# RGB basemap
results_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

# Classified image with palette
# Classes: 0=JUVI, 1=QUST, 2=ULCR, 7=GRSS, 8=WATR, 9=SHADE
# Need 10 colors for class values 0-9 (even though 3-6 are unused)
palette = [
    'FFFF00',    # 0: JUVI - Yellow
    '00FF00',    # 1: QUST - Green
    'FF00FF',    # 2: ULCR - Magenta
    '808080',    # 3: (unused) - Grey
    '808080',    # 4: (unused) - Grey
    '808080',    # 5: (unused) - Grey
    '808080',    # 6: (unused) - Grey
    '90EE90',    # 7: GRSS - Light green
    '0000FF',    # 8: WATR - Blue
    '000000'     # 9: SHADE - Black
]

results_map.addLayer(
    classified,
    {'min': 0, 'max': 9, 'palette': palette},
    'Classification 2022',
    True,
    0.7  # 70% opacity to see basemap through
)

# Add reference points
for class_num, name, color in [(0,'JUVI','yellow'),(1,'QUST','green'),(2,'ULCR','purple'),
                                (7,'GRSS','lightgreen'),(8,'WATR','blue'),(9,'SHADE','black')]:
    class_pts = all_points.filter(ee.Filter.eq('class', class_num))
    styled = class_pts.map(lambda f: f.set('style', {'color': color, 'pointShape': 'circle', 'pointSize': 3}))
    results_map.addLayer(styled.style(styleProperty='style'), {}, f'{name} points', False)

results_map.centerObject(CLBJ_Airshed, 13)
results_map

Map(center=[33.40123000139611, -97.57000034546598], controls=(WidgetControl(options=['position', 'transparent_…

## Discussion

### Comparison with Phase 1

Phase 2 improves on Phase 1 in several ways:

1. **Updated Data**: Uses 2022 BRDF-corrected reflectance (`HSI_REFL/002`) instead of legacy 2017 data
2. **Field-Based Training**: Builds training points directly from VST field data, avoiding temporal mismatch
3. **Transparent Workflow**: Shows the complete pipeline from field data download to classification
4. **Better Validation**: Reports out-of-bag error in addition to test set accuracy

However, accuracy may differ from Phase 1 due to:
- Different year (2022 vs 2017) - forest composition may have changed
- Different species composition - VST captures what was present in spring 2022
- Different number and distribution of training points
- BRDF correction in the 2022 data reduces sun angle effects

### Limitations Addressed in Phase 3

Phase 2 still has methodological limitations that Phase 3 will address:

1. **Spatial autocorrelation**: The random split places stems from the same plot in both training and test sets. Trees in the same plot are spectrally similar, inflating test accuracy. Phase 3 will use plot-level splitting.

2. **Class imbalance**: Some species have many more points than others, biasing the classifier toward common species. Phase 3 will balance class sizes or use class weights.

3. **Threshold-based non-tree classes**: The GRSS, WATR, and SHADE points rely on simple thresholds that may not capture all variability. Field validation would improve these classes.

4. **No cross-validation**: A single 80/20 split provides one accuracy estimate. Phase 3 could use k-fold cross-validation for more robust estimates.

5. **Hyperparameter tuning**: The random forest uses default parameters (100 trees). Phase 3 could optimize the number of trees, maximum depth, and other parameters.

Despite these limitations, Phase 2 demonstrates a complete, reproducible workflow for tree species classification using NEON AOP and field data in Google Earth Engine.

## Next Steps

To extend this tutorial:

1. **Export the classified image** using `ee.batch.Export.image.toDrive()` for use in GIS software
2. **Apply to other years** at CLBJ (2023, 2024, 2025) to track forest change
3. **Adapt to other NEON sites** in similar ecosystems
4. **Implement Phase 3 corrections** for more robust accuracy estimates
5. **Add feature importance analysis** to understand which bands contribute most to classification

For questions or feedback on this tutorial, contact NEON Data Skills or submit an issue to the tutorial repository.